In [1]:
%matplotlib inline

import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE" # tells OpenMP to not complain if it notices that two copies of OpenMP are loaded.

In [13]:
import requests
from torchcodec.decoders import VideoDecoder
from IPython.display import Video

def play_video(encoded_bytes):
    return Video(data=encoded_bytes.numpy().tobytes(),
                embed=True, width=640, height=360, mimetype="video/mp4")

# Video source: https://www.pexels.com/video/adorable-cats-on-the-lawn-4977395/
# Author: Altaf Shah.
url = "https://videos.pexels.com/video-files/4977395/4977395-hd_1920_1080_24fps.mp4"

response=requests.get(url, headers={'User-Agent':''})
if response.status_code!=200: raise RuntimeError(f"Failed to download video. {response.status_code=}.")

raw_video_bytes=response.content
decoder=VideoDecoder(raw_video_bytes)
frames=decoder.get_frames_in_range(0,60).data # get the first 60 frames
frame_rate=decoder.metadata.average_fps
print(f'{frames.shape=}, {frame_rate=}')

frames.shape=torch.Size([60, 3, 1080, 1920]), frame_rate=24.0


In [3]:
from torchcodec.encoders import VideoEncoder

print(f"{frames.shape=}, {frames.dtype=}")
print(f'{frame_rate=} fps') 
# input frames must be 4D torch.uint8 of shape (num_frames, num_channels, height, width) with values in [0,255] 
encoder=VideoEncoder(frames=frames, frame_rate=frame_rate) # frame_rate is the frame rate of input video
encoded_frames=encoder.to_tensor(format='mp4')
# play_video(encoded_frames)

frames.shape=torch.Size([60, 3, 1080, 1920]), frames.dtype=torch.uint8
frame_rate=24.0 fps


In [4]:
decoder_verify=VideoDecoder(encoded_frames)
decoded_frames=decoder_verify.get_frames_in_range(0, 60).data

print(f'Re-decoded video: {decoded_frames.shape=}')
print(f'Original frames: {frames.shape=}')

Re-decoded video: decoded_frames.shape=torch.Size([60, 3, 1080, 1920])
Original frames: frames.shape=torch.Size([60, 3, 1080, 1920])


In [5]:
from pathlib import Path

data_dirpath=Path('D:/results/temp')

# H.264 encoding
h264_output=data_dirpath/'h264.mp4'
encoder.to_file(h264_output, codec='libx264')

# H.265 encoding
hevc_output=data_dirpath/'hevc.mp4'
encoder.to_file(hevc_output, codec='hevc')

# Now let's use ffprobe to verify the codec used in the output files
import subprocess

for output, name in [(h264_output, 'h264_output'), (hevc_output, 'hevc_output')]:
    result=subprocess.run([
        "ffprobe", "-v",
        "error", "-select_streams",
        "v:0", "-show_entries",
        "stream=codec_name", "-of",
        "default=noprint_wrappers=1:nokey=1", output,
    ], capture_output=True, text=True)
    print(f'Codec used in {name}: {result.stdout.strip()}')

Codec used in h264_output: h264
Codec used in hevc_output: hevc


In [10]:
# Standard pixel format
yuv420_encoded_frames=encoder.to_tensor(format='mp4', codec='libx264', pixel_format='yuv420p')
# play_video(yuv420_encoded_frames)
print(f'{yuv420_encoded_frames.shape=}')

yuv420_encoded_frames.shape=torch.Size([2485963])


In [7]:
# High quality (low CRF)
high_quality_output=encoder.to_tensor(format='mp4', codec='libx264', crf=0)
# play_video(high_quality_output)

In [8]:
# Low quality (high CRF)
low_quality_output=encoder.to_tensor(format='mp4', codec='libx264', crf=50)
# play_video(low_quality_output)

In [9]:
# Fast encoding with a larger file size
fast_output='ultrafast.mp4'
encoder.to_file(fast_output, codec='libx264', preset='ultrafast')
print(f'Size of fast encoded file: {Path(fast_output).stat().st_size} bytes')

# Slow encoding for a smaller file size
slow_output='veryslow.mp4'
encoder.to_file(slow_output, codec='libx264', preset='veryslow')
print(f'Size of slow encoded file: {Path(slow_output).stat().st_size} bytes')

Size of fast encoded file: 7245618 bytes
Size of slow encoded file: 2112357 bytes
